In [1]:
from pathlib import Path  
from massipipe.pipeline import PipelineProcessor, find_datasets
from massipipe.utils import rgb_subset_from_hsi, read_envi, percentile_stretch_image, save_png
from massipipe.mosaic import convert_geotiff_to_8bit
from massipipe.quicklook import QuickLookProcessor
from rich import print

In [2]:
# Parameters
dataset_dir = Path('/home/mha114/data/seabee-minio/larvik/olbergholmen/aerial/hsi/20230830/massimal_larvik_olbergholmen_202308300959-south1_hsi')
output_base_dir = Path('/home/mha114/data/export')

rgb_wl_rededge = (700,550,460)

In [3]:
pipeline = PipelineProcessor(dataset_dir)

12:23:53 INFO: ----------------------------------------------------------------
12:23:53 INFO: File logging for massimal_larvik_olbergholmen_202308300959-south1_hsi initialized.
12:23:53 INFO: Found directory 0_raw - processing from raw files.


In [8]:
pipeline.config.radiance_gc.overwrite = True
pipeline.config.radiance_gc.smooth_spectra = True
pipeline.config.radiance_gc_rgb.overwrite = True
pipeline.config.general.rgb_wl = rgb_wl_rededge

print(pipeline.config)

MassipipeOptions(
    general=MpGeneral(rgb_wl=(700, 550, 460)),
    quicklook=MpQuickLook(create=True, overwrite=False, percentiles=None),
    imu_data=MpImuData(create=True, overwrite=False),
    geotransform=MpGeoTransform(
        create=True,
        overwrite=True,
        camera_opening_angle_deg=36.5,
        pitch_offset_deg=0.0,
        roll_offset_deg=0.0,
        altitude_offset_m=0.0,
        utm_x_offset_m=0.0,
        utm_y_offset_m=0.0,
        assume_square_pixels=True
    ),
    radiance=MpRadiance(
        create=True,
        overwrite=False,
        set_saturated_pixels_to_zero=True,
        add_envi_mapinfo_to_header=True,
        add_irradiance_to_header=True
    ),
    radiance_rgb=MpRadianceRgb(create=True, overwrite=False),
    radiance_gc=MpRadianceGc(
        create=True,
        overwrite=True,
        smooth_spectra=True,
        subtract_dark_spec=False,
        set_negative_values_to_zero=False,
        reference_image_numbers=[3, 8],
        reference_image_ranges=[(500, 1300, 0, 550), (1250, 1650, 0, 900)]
    ),
    radiance_gc_rgb=MpRadianceGcRgb(create=True, overwrite=True),
    irradiance=MpIrradiance(create=True, overwrite=False),
    reflectance=MpReflectance(
        create=True,
        overwrite=False,
        wl_min=400.0,
        wl_max=930.0,
        conv_irrad_with_gauss=True,
        fwhm_irrad_smooth=3.5,
        smooth_spectra=False,
        refl_from_mean_irrad=False
    ),
    reflectance_gc=MpReflectanceGc(create=True, overwrite=False, smooth_spectra=True, method='from_rad_gc'),
    reflectance_gc_rgb=MpReflectanceGcRgb(create=True, overwrite=False),
    mosaic=MpMosaic(
        overview_factors=[2, 4, 8, 16, 32],
        radiance_gc_rgb=MpMosaicCreateOverwrite(create=True, overwrite=True),
        reflectance_gc_rgb=MpMosaicCreateOverwrite(create=False, overwrite=True)
    )
)

In [5]:
# pipeline.glint_correct_radiance_images()

In [9]:
pipeline.create_glint_corrected_radiance_rgb_geotiff()

12:34:35 INFO: ---- EXPORTING RGB GEOTIFF FOR GLINT CORRECTED RADIANCE ----
12:34:35 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_000_radiance_gc.bip.hdr.
12:34:37 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_001_radiance_gc.bip.hdr.
12:34:40 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_002_radiance_gc.bip.hdr.
12:34:40 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_003_radiance_gc.bip.hdr.
12:34:43 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_004_radiance_gc.bip.hdr.
12:34:45 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_005_radiance_gc.bip.hdr.
12:34:48 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-south1_hsi_006_radiance_gc.bip.hdr.
12:34:50 INFO: Exporting RGB GeoTIFF from massimal_larvik_olbergholmen_202308300959-

In [10]:
# Export PNGs and 8-bit GeoTiffs
output_dir = output_base_dir / pipeline.dataset_base_name
output_dir.mkdir(exist_ok=True)

for rad_gc_im, rad_gc_im_rgb in zip(pipeline.rad_gc_im_paths, pipeline.rad_gc_rgb_im_paths):
    image,wl,metadata = read_envi(rad_gc_im)
    
    # Create percentile stretched version for PNG
    image[image<0] = 0
    image_rgb, rgb_wl = rgb_subset_from_hsi(image,wl,rgb_target_wl=rgb_wl_rededge)
    image_rgb = percentile_stretch_image(image_rgb)
    
    png_path = output_dir / (rad_gc_im.name.split(".")[0] + '.png')    
    save_png(image_rgb,png_path)
    
    # Convert existing GeoTIFF to 8-bit
    geotiff_8bit_path = output_dir / rad_gc_im_rgb.name
    convert_geotiff_to_8bit(rad_gc_im_rgb,geotiff_8bit_path)


12:35:19 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_000_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:19 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_001_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:19 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_002_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:19 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_003_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:20 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_004_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:20 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_005_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:20 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_006_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:20 INFO: Converting massimal_larvik_olbergholmen_202308300959-south1_hsi_007_radiance_gc_rgb.tiff to 8-bit geotiff
12:35:20 INFO: Converting massim